# ArUco-basierte Treffererkennung und Spielumgebung

Dieses Notebook realisiert die markerbasierte Positionsbestimmung der Fahrzeuge sowie die softwareseitige Trefferlogik.

## Funktionen
- Marker-Erkennung
- Positions- und Richtungsbestimmung
- Hitbox-Berechnung
- Spielzustandsverwaltung
- Visualisierung mit PyGame


In [ ]:
import time
import cv2
import numpy as np
from cv2 import aruco
import pygame
import subprocess
import platform
from dataclasses import dataclass, field
from typing import Optional, Tuple

# =============================
# ArUco Konfiguration
# =============================
CALIB_FILE = "camera_calib_neu.npz"

ID_SHOOTER = 0
ID_TARGET  = 1

MARKER_LENGTH_M = 0.056   # 56 mm
HIT_RADIUS_M    = 0.11    # 11 cm Hitbox-Radius

CAM_INDEX = 0

SHOW_AXES = True
AXIS_LEN_M = 0.03

FRONT_IS_MINUS_Y = False

SHOW_CV_WINDOW = True

SHOT_OVERLAY_S = 1.5

# Visualisierung der Schusslinie
RAY_VIS_MAX_M = 2.5
RAY_SAMPLES   = 25

# =============================
# Netzwerk / Hotspot / Verbindung
# =============================
# HIER DEINE IPs EINTRAGEN
TANK1_IP = "XXX"
TANK2_IP = "XXX"

CONN_CHECK_INTERVAL_S = 1.0

PING_TIMEOUT_MS = 250

# =============================
# Pygame UI Konfiguration
# =============================
WIDTH, HEIGHT = 1280, 720
FPS = 60

BLUE = (59, 130, 246)
RED  = (239, 68, 68)
BG   = (11, 15, 25)
WHITE= (229, 231, 235)
GRAY = (150, 160, 170)
GREEN= (34, 197, 94)

PH_HOME = "HOME"
PH_SETTINGS = "SETTINGS"
PH_COUNTDOWN = "COUNTDOWN"
PH_INGAME = "INGAME"
PH_GAMEOVER = "GAMEOVER"

XBOX_LB_BTN = 4
XBOX_RB_BTN = 5

# =============================
# ArUco Helper
# =============================
def rvec_to_R(rvec):
    R, _ = cv2.Rodrigues(rvec)
    return R

def forward_vector_cam_from_rvec(rvec, front_is_minus_y: bool):
    R = rvec_to_R(rvec)
    forward_marker = np.array([0.0, -1.0 if front_is_minus_y else 1.0, 0.0], dtype=np.float64)
    f = R @ forward_marker
    return f / (np.linalg.norm(f) + 1e-9)

def ray_sphere_hit(o, fwd, p, hit_radius):
    """
    Ray-Sphere-Test:
      Ray:    x(t) = o + t*fwd, t >= 0
      Sphere: ||x - p|| = hit_radius

    Rückgabe:
      hit, d_center, angle_deg, theta_deg, impact_point, closest_point, miss_dist, t_closest
    """
    fwd = fwd / (np.linalg.norm(fwd) + 1e-9)
    to_target = p - o
    d_center = float(np.linalg.norm(to_target))

    if d_center < 1e-9:
        return True, 0.0, 0.0, 90.0, p.copy(), p.copy(), 0.0, 0.0

    to_unit = to_target / d_center
    cosang = float(np.dot(fwd, to_unit))
    cosang = np.clip(cosang, -1.0, 1.0)
    angle_deg = float(np.degrees(np.arccos(cosang)))
    theta_deg = float(np.degrees(np.arctan(hit_radius / d_center)))

    t_closest = float(np.dot(to_target, fwd))
    if t_closest < 0:
        closest_point = o.copy()
        miss_dist = float(np.linalg.norm(p - closest_point))
        return False, d_center, angle_deg, theta_deg, None, closest_point, miss_dist, t_closest

    closest_point = o + t_closest * fwd
    miss_dist = float(np.linalg.norm(p - closest_point))

    if miss_dist > hit_radius:
        return False, d_center, angle_deg, theta_deg, None, closest_point, miss_dist, t_closest

    thc = float(np.sqrt(max(hit_radius**2 - miss_dist**2, 0.0)))
    t_entry = t_closest - thc
    if t_entry < 0:
        t_entry = t_closest + thc
        if t_entry < 0:
            return False, d_center, angle_deg, theta_deg, None, closest_point, miss_dist, t_closest

    impact_point = o + t_entry * fwd
    return True, d_center, angle_deg, theta_deg, impact_point, closest_point, miss_dist, t_closest

def put_text(frame, text, org, scale=0.7):
    cv2.putText(frame, text, org, cv2.FONT_HERSHEY_SIMPLEX, scale, (0,0,0), 3, cv2.LINE_AA)
    cv2.putText(frame, text, org, cv2.FONT_HERSHEY_SIMPLEX, scale, (255,255,255), 1, cv2.LINE_AA)

def project_cam_points(points_3d, camera_matrix, dist_coeffs):
    points_3d = np.asarray(points_3d, dtype=np.float32).reshape(-1, 3)
    img_pts, _ = cv2.projectPoints(
        points_3d,
        np.zeros((3, 1), dtype=np.float32),
        np.zeros((3, 1), dtype=np.float32),
        camera_matrix,
        dist_coeffs
    )
    return np.round(img_pts.reshape(-1, 2)).astype(int)

def draw_hitbox_circle(frame, camera_matrix, dist_coeffs, target_tvec, hit_radius_m,
                       color=(0, 255, 0), thickness=2):
    center = target_tvec.reshape(3).astype(np.float32)

    circle_pts_3d = []
    for a in np.linspace(0, 2*np.pi, 72, endpoint=False):
        x = center[0] + hit_radius_m * np.cos(a)
        y = center[1] + hit_radius_m * np.sin(a)
        z = center[2]
        circle_pts_3d.append([x, y, z])

    img_pts = project_cam_points(circle_pts_3d, camera_matrix, dist_coeffs)
    if len(img_pts) >= 2:
        cv2.polylines(frame, [img_pts.reshape(-1, 1, 2)], True, color, thickness, cv2.LINE_AA)

    c = project_cam_points([center], camera_matrix, dist_coeffs)[0]
    cv2.circle(frame, tuple(c), 3, color, -1, cv2.LINE_AA)

def draw_projected_polyline(frame, points_3d, camera_matrix, dist_coeffs, color, thickness=2):
    pts_2d = project_cam_points(points_3d, camera_matrix, dist_coeffs)
    if len(pts_2d) >= 2:
        cv2.polylines(frame, [pts_2d.reshape(-1, 1, 2)], False, color, thickness, cv2.LINE_AA)

def draw_shot_visualization(frame, camera_matrix, dist_coeffs, o, fwd, p,
                            hit, impact_point, closest_point, hit_radius_m):
    fwd = fwd / (np.linalg.norm(fwd) + 1e-9)

    dist_to_target = float(np.linalg.norm(p - o))
    if hit and impact_point is not None:
        ray_len = min(RAY_VIS_MAX_M, max(0.25, float(np.linalg.norm(impact_point - o)) + 0.05))
    else:
        ray_len = min(RAY_VIS_MAX_M, max(0.4, dist_to_target + 0.2))

    ts = np.linspace(0.0, ray_len, RAY_SAMPLES)
    ray_points = [o + t * fwd for t in ts]

    ray_color = (0, 255, 0) if hit else (0, 180, 255)
    draw_projected_polyline(frame, ray_points, camera_matrix, dist_coeffs, ray_color, 2)

    o_2d = project_cam_points([o], camera_matrix, dist_coeffs)[0]
    p_2d = project_cam_points([p], camera_matrix, dist_coeffs)[0]
    cv2.circle(frame, tuple(o_2d), 5, (255, 255, 0), -1, cv2.LINE_AA)
    cv2.circle(frame, tuple(p_2d), 5, (0, 0, 255), -1, cv2.LINE_AA)

    arrow_end = o + min(0.18, ray_len) * fwd
    arrow_2d = project_cam_points([arrow_end], camera_matrix, dist_coeffs)[0]
    cv2.arrowedLine(frame, tuple(o_2d), tuple(arrow_2d), (255, 255, 0), 2, cv2.LINE_AA, tipLength=0.2)

    if hit and impact_point is not None:
        ip_2d = project_cam_points([impact_point], camera_matrix, dist_coeffs)[0]
        cv2.circle(frame, tuple(ip_2d), 7, (0, 255, 0), -1, cv2.LINE_AA)
        cv2.circle(frame, tuple(ip_2d), 12, (0, 255, 0), 2, cv2.LINE_AA)
        put_text(frame, "impact", (ip_2d[0] + 8, ip_2d[1] - 8), 0.5)
    elif closest_point is not None:
        cp_2d = project_cam_points([closest_point], camera_matrix, dist_coeffs)[0]
        cv2.circle(frame, tuple(cp_2d), 6, (0, 180, 255), -1, cv2.LINE_AA)
        cv2.line(frame, tuple(cp_2d), tuple(p_2d), (0, 180, 255), 1, cv2.LINE_AA)
        put_text(frame, "closest", (cp_2d[0] + 8, cp_2d[1] - 8), 0.5)

# =============================
# Netzwerk Helper
# =============================
def is_host_reachable(ip: str, timeout_ms: int = PING_TIMEOUT_MS) -> bool:
    """
    Prüft per Ping, ob ein Gerät mit der gegebenen IP erreichbar ist.
    Für Windows und Linux/macOS geeignet.
    """
    system = platform.system().lower()

    if system == "windows":
        cmd = ["ping", "-n", "1", "-w", str(timeout_ms), ip]
    else:
        timeout_s = max(1, int(np.ceil(timeout_ms / 1000.0)))
        cmd = ["ping", "-c", "1", "-W", str(timeout_s), ip]

    try:
        result = subprocess.run(
            cmd,
            stdout=subprocess.DEVNULL,
            stderr=subprocess.DEVNULL
        )
        return result.returncode == 0
    except Exception:
        return False

# =============================
# Game State
# =============================
@dataclass
class Team:
    name: str
    color: Tuple[int,int,int]
    lives: int

@dataclass
class Config:
    duration_s: int = 5*60
    lives: int = 5
    fire_rate_hz: float = 2.0

@dataclass
class Conn:
    connected: bool = False
    last_seen: float = 0.0

@dataclass
class Game:
    phase: str = PH_HOME
    cfg: Config = field(default_factory=Config)
    team1: Team = field(default_factory=lambda: Team("Team A", BLUE, 5))
    team2: Team = field(default_factory=lambda: Team("Team B", RED, 5))
    conn1: Conn = field(default_factory=Conn)
    conn2: Conn = field(default_factory=Conn)

    can_move: bool = False
    remaining_s: int = 0
    countdown: float = 0.0
    winner: Optional[int] = None

G = Game()

def fmt_time(s: int) -> str:
    m = s // 60
    r = s % 60
    return f"{m:02d}:{r:02d}"

def set_connected(team_id: int, connected: bool):
    if team_id == 1:
        c = G.conn1
    elif team_id == 2:
        c = G.conn2
    else:
        return

    c.connected = bool(connected)
    if connected:
        c.last_seen = time.time()

def register_hit(target_id: int):
    """target_id=1 => Team1 verliert Leben, target_id=2 => Team2 verliert Leben"""
    if G.phase != PH_INGAME:
        return
    team = G.team1 if target_id == 1 else G.team2
    team.lives = max(0, team.lives - 1)
    if team.lives == 0:
        end_game(2 if target_id == 1 else 1)

def end_game(winner: Optional[int]):
    G.phase = PH_GAMEOVER
    G.can_move = False
    G.winner = winner

def apply_settings(duration_s: int, lives: int, fire_rate: float, name1: str, name2: str):
    G.cfg.duration_s = duration_s
    G.cfg.lives = lives
    G.cfg.fire_rate_hz = fire_rate

    G.team1.name = name1.strip() or "Team A"
    G.team2.name = name2.strip() or "Team B"

    G.team1.lives = lives
    G.team2.lives = lives

# =============================
# UI Widgets
# =============================
class Button:
    def __init__(self, rect, text, on_click, color=(60, 120, 200)):
        self.rect = pygame.Rect(rect)
        self.text = text
        self.on_click = on_click
        self.color = color

    def draw(self, screen, font):
        pygame.draw.rect(screen, self.color, self.rect, border_radius=12)
        lbl = font.render(self.text, True, (255,255,255))
        screen.blit(lbl, lbl.get_rect(center=self.rect.center))

    def handle(self, ev):
        if ev.type == pygame.MOUSEBUTTONDOWN and ev.button == 1:
            if self.rect.collidepoint(ev.pos):
                self.on_click()

class CycleOption:
    def __init__(self, label, options, value_index=0):
        self.label = label
        self.options = options
        self.idx = value_index

    @property
    def value(self):
        return self.options[self.idx][1]

    @property
    def display(self):
        return self.options[self.idx][0]

    def cycle(self, delta):
        self.idx = (self.idx + delta) % len(self.options)

class TextField:
    def __init__(self, rect, text=""):
        self.rect = pygame.Rect(rect)
        self.text = text
        self.active = False

    def handle(self, ev):
        if ev.type == pygame.MOUSEBUTTONDOWN and ev.button == 1:
            self.active = self.rect.collidepoint(ev.pos)

        if self.active and ev.type == pygame.KEYDOWN:
            if ev.key == pygame.K_BACKSPACE:
                self.text = self.text[:-1]
            elif ev.key == pygame.K_RETURN:
                self.active = False
            else:
                if len(ev.unicode) == 1 and len(self.text) < 18:
                    self.text += ev.unicode

    def draw(self, screen, font):
        col = (255,255,255) if self.active else (200,200,200)
        pygame.draw.rect(screen, (20, 26, 40), self.rect, border_radius=10)
        pygame.draw.rect(screen, col, self.rect, 2, border_radius=10)
        lbl = font.render(self.text, True, WHITE)
        screen.blit(lbl, (self.rect.x+10, self.rect.y+8))

def draw_panel(screen, rect):
    surf = pygame.Surface((rect.width, rect.height), pygame.SRCALPHA)
    surf.fill((255,255,255,18))
    pygame.draw.rect(surf, (255,255,255,40), surf.get_rect(), 1, border_radius=18)
    screen.blit(surf, rect.topleft)

# =============================
# MAIN
# =============================
def main():
    # Camera & ArUco
    cal = np.load(CALIB_FILE)
    camera_matrix = cal["camera_matrix"]
    dist_coeffs = cal["dist_coeffs"]

    aruco_dict = aruco.getPredefinedDictionary(aruco.DICT_4X4_50)
    params = aruco.DetectorParameters()
    detector = aruco.ArucoDetector(aruco_dict, params)

    cap = cv2.VideoCapture(CAM_INDEX)
    if not cap.isOpened():
        raise RuntimeError(f"Kamera {CAM_INDEX} konnte nicht geöffnet werden.")

    # Pygame
    pygame.init()
    pygame.display.set_caption("Panzer-Spiel UI")
    screen = pygame.display.set_mode((WIDTH, HEIGHT))
    clock = pygame.time.Clock()

    font_big = pygame.font.SysFont("Segoe UI", 64, bold=True)
    font_mid = pygame.font.SysFont("Segoe UI", 32, bold=True)
    font_sm  = pygame.font.SysFont("Segoe UI", 22)

    # Gamepad
    pygame.joystick.init()
    joy = None
    if pygame.joystick.get_count() > 0:
        joy = pygame.joystick.Joystick(0)
        joy.init()
        print("Gamepad verbunden:", joy.get_name())
        print("Hinweis: LB/RB Button-Index ggf. anders. Debug: JOYBUTTONDOWN ausgeben, falls nötig.")
    else:
        print("Kein Gamepad gefunden – Shoot per SPACE bleibt aktiv.")

    opt_time = CycleOption("Spielzeit", [("5 min", 300), ("10 min", 600), ("15 min", 900), ("20 min", 1200)], 0)
    opt_lives= CycleOption("Leben", [("1",1),("3",3),("5",5),("7",7),("10",10)], 2)
    opt_rate = CycleOption("Schussrate", [("0.5 /s",0.5),("1 /s",1.0),("2 /s",2.0),("3 /s",3.0),("5 /s",5.0)], 2)

    tf1 = TextField((WIDTH//2-220, 320, 440, 44), "Team A")
    tf2 = TextField((WIDTH//2-220, 380, 440, 44), "Team B")

    apply_settings(opt_time.value, opt_lives.value, opt_rate.value, tf1.text, tf2.text)

    buttons = []

    last_second = time.time()

    # Shot rate
    last_shot_t = 0.0
    last_shot_result = None
    last_shot_info = ""
    last_shot_overlay_t = 0.0

    vis_shot_hit = None
    vis_impact_point = None
    vis_closest_point = None

    # Netzwerkprüfung
    last_conn_check_t = 0.0

    running = True
    while running:
        dt = clock.tick(FPS) / 1000.0
        now = time.time()

        # ===========================
        # Netzwerkstatus prüfen
        # ===========================
        if now - last_conn_check_t >= CONN_CHECK_INTERVAL_S:
            last_conn_check_t = now

            tank1_ok = is_host_reachable(TANK1_IP)
            tank2_ok = is_host_reachable(TANK2_IP)

            set_connected(1, tank1_ok)
            set_connected(2, tank2_ok)

        # ===========================
        # 1) Kamera lesen + ArUco detektieren
        # ===========================
        poses = {}
        ret, frame = cap.read()
        if ret:
            corners, ids, _ = detector.detectMarkers(frame)

            put_text(
                frame,
                f"LB/RB oder SPACE=shoot | shooter={ID_SHOOTER} target={ID_TARGET} | "
                f"L={MARKER_LENGTH_M*1000:.0f}mm hitbox_r={HIT_RADIUS_M*1000:.0f}mm",
                (10, 25), 0.6
            )

            put_text(
                frame,
                f"Netzwerk: T1={'ON' if G.conn1.connected else 'OFF'} ({TANK1_IP}) | "
                f"T2={'ON' if G.conn2.connected else 'OFF'} ({TANK2_IP})",
                (10, 150), 0.6
            )

            if ids is not None:
                ids_flat = ids.flatten().astype(int)
                aruco.drawDetectedMarkers(frame, corners, ids)

                rvecs, tvecs, _ = aruco.estimatePoseSingleMarkers(
                    corners, MARKER_LENGTH_M, camera_matrix, dist_coeffs
                )

                for i, mid in enumerate(ids_flat):
                    rvec = rvecs[i].reshape(3, 1)
                    tvec = tvecs[i].reshape(3, 1)
                    poses[int(mid)] = (rvec, tvec)

                    if SHOW_AXES:
                        cv2.drawFrameAxes(frame, camera_matrix, dist_coeffs, rvec, tvec, AXIS_LEN_M)

                    color = (0, 255, 0) if int(mid) == ID_TARGET else (150, 150, 150)
                    draw_hitbox_circle(
                        frame,
                        camera_matrix,
                        dist_coeffs,
                        tvec,
                        HIT_RADIUS_M,
                        color=color,
                        thickness=2
                    )

                if ID_SHOOTER in poses and ID_TARGET in poses:
                    rvec_s, tvec_s = poses[ID_SHOOTER]
                    rvec_t, tvec_t = poses[ID_TARGET]

                    o = tvec_s.reshape(3)
                    p = tvec_t.reshape(3)
                    fwd = forward_vector_cam_from_rvec(rvec_s, FRONT_IS_MINUS_Y)

                    hit_preview, d_preview, ang_preview, theta_preview, impact_preview, closest_preview, miss_preview, _ = ray_sphere_hit(
                        o, fwd, p, HIT_RADIUS_M
                    )

                    draw_shot_visualization(
                        frame,
                        camera_matrix,
                        dist_coeffs,
                        o,
                        fwd,
                        p,
                        vis_shot_hit if vis_shot_hit is not None else hit_preview,
                        vis_impact_point if vis_impact_point is not None else impact_preview,
                        vis_closest_point if vis_closest_point is not None else closest_preview,
                        HIT_RADIUS_M
                    )

                    put_text(
                        frame,
                        f"preview: {'HIT' if hit_preview else 'MISS'} | d={d_preview:.2f}m angle={ang_preview:.1f}deg tol={theta_preview:.1f}deg",
                        (10, 95),
                        0.6
                    )

                    if not hit_preview:
                        put_text(frame, f"miss_dist={miss_preview*100:.1f}cm", (10, 125), 0.6)

            if last_shot_result is not None and (now - last_shot_overlay_t) < SHOT_OVERLAY_S:
                status = "HIT" if last_shot_result else "NO HIT"
                put_text(frame, f"SHOT: {status} | {last_shot_info}", (10, 60), 0.85)

            if SHOW_CV_WINDOW:
                cv2.imshow("ArUco Live", frame)
                cv2.waitKey(1)

        # ===========================
        # 2) Pygame Events + UI Input
        # ===========================
        shoot_edge = False
        for ev in pygame.event.get():
            if ev.type == pygame.QUIT:
                running = False

            if ev.type == pygame.KEYDOWN:
                if ev.key == pygame.K_ESCAPE:
                    running = False
                if ev.key == pygame.K_F11:
                    pygame.display.toggle_fullscreen()
                if ev.key == pygame.K_SPACE:
                    shoot_edge = True

            if G.phase == PH_SETTINGS:
                tf1.handle(ev)
                tf2.handle(ev)

            for b in buttons:
                b.handle(ev)

            if ev.type == pygame.JOYBUTTONDOWN:
                if ev.button in (XBOX_LB_BTN, XBOX_RB_BTN):
                    shoot_edge = True

            if G.phase == PH_SETTINGS and ev.type == pygame.MOUSEBUTTONDOWN and ev.button == 1:
                mx, my = ev.pos

                def arrow_hit(y):
                    left = pygame.Rect(WIDTH//2-260, y, 30, 30)
                    right= pygame.Rect(WIDTH//2+230, y, 30, 30)
                    return left.collidepoint(mx,my), right.collidepoint(mx,my)

                l, r = arrow_hit(200)
                if l: opt_time.cycle(-1)
                if r: opt_time.cycle(+1)

                l, r = arrow_hit(240)
                if l: opt_lives.cycle(-1)
                if r: opt_lives.cycle(+1)

                l, r = arrow_hit(280)
                if l: opt_rate.cycle(-1)
                if r: opt_rate.cycle(+1)

        # ===========================
        # 3) Hold to fire + Schussratenlimit
        # ===========================
        shoot_held = False
        if joy is not None:
            try:
                shoot_held = bool(joy.get_button(XBOX_LB_BTN) or joy.get_button(XBOX_RB_BTN))
            except pygame.error:
                shoot_held = False

        shoot_pressed = shoot_edge or shoot_held

        # ===========================
        # 4) State Updates
        # ===========================
        if G.phase == PH_COUNTDOWN:
            G.countdown -= dt
            if G.countdown <= 0:
                G.phase = PH_INGAME
                G.can_move = True
                G.remaining_s = G.cfg.duration_s
                last_second = now

        if G.phase == PH_INGAME:
            if now - last_second >= 1.0:
                last_second += 1.0
                G.remaining_s -= 1
                if G.remaining_s <= 0:
                    if G.team1.lives == G.team2.lives:
                        end_game(None)
                    else:
                        end_game(1 if G.team1.lives > G.team2.lives else 2)

        # ===========================
        # 5) Schuss-Logik (rate-limited)
        # ===========================
        if shoot_pressed and G.phase == PH_INGAME:
            fire_rate = max(0.1, float(G.cfg.fire_rate_hz))
            min_interval = 1.0 / fire_rate

            if (now - last_shot_t) >= min_interval:
                last_shot_t = now

                if ID_SHOOTER in poses and ID_TARGET in poses:
                    rvec_s, tvec_s = poses[ID_SHOOTER]
                    rvec_t, tvec_t = poses[ID_TARGET]

                    o = tvec_s.reshape(3)
                    p = tvec_t.reshape(3)
                    fwd = forward_vector_cam_from_rvec(rvec_s, FRONT_IS_MINUS_Y)

                    hit, d, ang, theta, impact_point, closest_point, miss_dist, _ = ray_sphere_hit(
                        o, fwd, p, HIT_RADIUS_M
                    )

                    last_shot_result = hit
                    if hit:
                        last_shot_info = f"d={d:.2f}m angle={ang:.1f}deg tol={theta:.1f}deg"
                    else:
                        last_shot_info = f"d={d:.2f}m angle={ang:.1f}deg tol={theta:.1f}deg miss={miss_dist*100:.1f}cm"
                    last_shot_overlay_t = now

                    vis_shot_hit = hit
                    vis_impact_point = impact_point
                    vis_closest_point = closest_point

                    if hit:
                        register_hit(ID_TARGET)

                else:
                    last_shot_result = False
                    last_shot_info = "missing shooter/target marker"
                    last_shot_overlay_t = now
                    vis_shot_hit = False
                    vis_impact_point = None
                    vis_closest_point = None

        # ===========================
        # 6) Draw UI
        # ===========================
        screen.fill(BG)
        buttons = []

        if G.phase == PH_HOME:
            title = font_big.render("Panzer-Spiel", True, WHITE)
            screen.blit(title, title.get_rect(center=(WIDTH//2, 160)))

            def go_settings():
                G.phase = PH_SETTINGS

            buttons.append(Button((WIDTH//2-140, 260, 280, 54), "Zum Spielmenü", go_settings, color=(59,130,246)))

        elif G.phase == PH_SETTINGS:
            draw_panel(screen, pygame.Rect(WIDTH//2-320, 120, 640, 520))
            screen.blit(font_mid.render("Voreinstellungen", True, WHITE), (WIDTH//2-190, 140))

            def draw_cycle(y, opt: CycleOption):
                screen.blit(font_sm.render(f"{opt.label}:", True, GRAY), (WIDTH//2-220, y))
                screen.blit(font_sm.render(opt.display, True, WHITE), (WIDTH//2-40, y))
                pygame.draw.polygon(screen, WHITE, [(WIDTH//2-245, y+8),(WIDTH//2-235, y+15),(WIDTH//2-245, y+22)])
                pygame.draw.polygon(screen, WHITE, [(WIDTH//2+245, y+8),(WIDTH//2+255, y+15),(WIDTH//2+245, y+22)])

            draw_cycle(200, opt_time)
            draw_cycle(240, opt_lives)
            draw_cycle(280, opt_rate)

            screen.blit(font_sm.render("Team 1:", True, GRAY), (WIDTH//2-300, 330))
            screen.blit(font_sm.render("Team 2:", True, GRAY), (WIDTH//2-300, 390))
            tf1.draw(screen, font_sm)
            tf2.draw(screen, font_sm)

            def conn_card(x, y, team: Team, conn: Conn, ip: str):
                r = pygame.Rect(x, y, 280, 90)
                draw_panel(screen, r)
                status = "Verbunden" if conn.connected else "Nicht verbunden"
                col = GREEN if conn.connected else (244, 63, 94)
                screen.blit(font_sm.render(team.name, True, team.color), (x+12, y+10))
                screen.blit(font_sm.render(status, True, col), (x+12, y+38))
                screen.blit(font_sm.render(ip, True, GRAY), (x+12, y+62))

            conn_card(WIDTH//2-300, 450, G.team1, G.conn1, TANK1_IP)
            conn_card(WIDTH//2+20, 450, G.team2, G.conn2, TANK2_IP)

            def apply():
                apply_settings(opt_time.value, opt_lives.value, opt_rate.value, tf1.text, tf2.text)

            def start():
                apply()
                G.phase = PH_COUNTDOWN
                G.can_move = False
                G.winner = None
                G.countdown = 3.0
                G.team1.lives = G.cfg.lives
                G.team2.lives = G.cfg.lives

            def back():
                G.phase = PH_HOME

            buttons.append(Button((WIDTH//2-280, 560, 180, 50), "Übernehmen", apply, color=(59,130,246)))
            buttons.append(Button((WIDTH//2-90, 560, 180, 50), "Spiel starten", start, color=(34,197,94)))
            buttons.append(Button((WIDTH//2+100, 560, 180, 50), "Zurück", back, color=(107,114,128)))

        elif G.phase == PH_COUNTDOWN:
            screen.blit(font_mid.render("Spiel startet in...", True, WHITE), (WIDTH//2-150, 180))
            n = max(0, int(G.countdown) + 1)
            screen.blit(font_big.render(str(n), True, WHITE), (WIDTH//2-20, 260))

        elif G.phase == PH_INGAME:
            screen.blit(font_big.render(fmt_time(G.remaining_s), True, WHITE), (WIDTH//2-110, 40))

            l1 = font_mid.render(f"{G.team1.name}:  ❤ {G.team1.lives}", True, G.team1.color)
            l2 = font_mid.render(f"{G.team2.name}:  ❤ {G.team2.lives}", True, G.team2.color)
            screen.blit(l1, (80, 140))
            screen.blit(l2, (80, 190))

            net1 = "ONLINE" if G.conn1.connected else "OFFLINE"
            net2 = "ONLINE" if G.conn2.connected else "OFFLINE"

            screen.blit(font_sm.render(f"{G.team1.name}: {net1} ({TANK1_IP})", True, GRAY), (80, 250))
            screen.blit(font_sm.render(f"{G.team2.name}: {net2} ({TANK2_IP})", True, GRAY), (80, 280))

            hint = font_sm.render(
                f"LB/RB oder SPACE=shoot | fire_rate={G.cfg.fire_rate_hz}/s | can_move={G.can_move} | F11=Fullscreen",
                True, GRAY
            )
            screen.blit(hint, (80, HEIGHT-40))

        elif G.phase == PH_GAMEOVER:
            if G.winner is None:
                msg = "Unentschieden"
                col = WHITE
            else:
                team = G.team1 if G.winner == 1 else G.team2
                msg = f"{team.name} gewinnt!"
                col = team.color

            screen.blit(font_big.render("GAME OVER", True, WHITE), (WIDTH//2-180, 160))
            screen.blit(font_mid.render(msg, True, col), (WIDTH//2-220, 260))

            def back_to_menu():
                G.phase = PH_SETTINGS
                G.can_move = False
                G.winner = None
                G.team1.lives = G.cfg.lives
                G.team2.lives = G.cfg.lives

            buttons.append(Button((WIDTH//2-120, 360, 240, 54), "Zurück ins Menü", back_to_menu, color=(59,130,246)))

        for b in buttons:
            b.draw(screen, font_sm)

        pygame.display.flip()

    cap.release()
    if SHOW_CV_WINDOW:
        cv2.destroyAllWindows()
    pygame.quit()

if __name__ == "__main__":
    main()